In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
import matplotlib
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable
from scipy.interpolate import CubicSpline
import matplotlib.colors as mcolors

# from generate_colormap import *


from functools import partial
import starsim

from astropy.io import fits

from scipy.optimize import curve_fit

In [2]:
pathdata = '/home/sophie-stucki/Documents/starsim_simulations/nina_spec/'
path_grid="/home/sophie-stucki/starsim/starsim/gif/"
conf_file_path = "/home/sophie-stucki/sunsim/conf/starsim_test_instr.conf"

In [3]:
def simulation_init(type_list, spot_size_list, latitude_list, longitude_list, conf_file_path, dT_fc=None):
    """
    Iniialization of the starsim simulation for a particular set of parameters

    Params:
            -Q: facular_area_ratio
    Return:
            - ss: starsim object

    """
    
    #create the starsim object
    ss=starsim.StarSim(conf_file_path=conf_file_path)


    #set the Q parameter
    ss.active_region_types=type_list

    if dT_fc !=None:
        ss.facula_T_contrast = dT_fc

    #initialize the spot
    overlap=True

    #TODO: more modulable
    Nspots=len(spot_size_list)
    ss.spot_map=np.zeros([Nspots,8])
    for j in range(Nspots):
        ss.spot_map[j][0]=type_list[j]
        ss.spot_map[j][2]=200#lifetime spot
        ss.spot_map[j][1]=0#appearance time
        ss.spot_map[j][3]=latitude_list[j]#latitude (degrees) [0,180]
        ss.spot_map[j][4]=longitude_list[j]#longitude (degrees)	[0,360]
        ss.spot_map[j][5]=spot_size_list[j]#spot size (degrees)
    return ss

In [4]:
type_list = [0]
spot_size_list = [10]
latitude_list = [90]
longitude_list = [250]

period = 25
periods_nbr = 0.8
point_nbr = 1
bb_ratio = 1
t=np.linspace(0,period *periods_nbr,int(period *periods_nbr*point_nbr))

In [5]:
ss_mps = simulation_init(type_list, spot_size_list, latitude_list, longitude_list, conf_file_path)
ss_mps.compute_forward(observables=['rv'],t=t)


ELSE
ELSE
Limb extrapolation: constant for mu <  0.1
Limb extrapolation: constant for mu <  0.1
Limb extrapolation: constant for mu <  0.1
Limb extrapolation: constant for mu <  0.1
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Spot: Add CFIST
Sp

/home/sophie-stucki/starsim/starsim/spectra.py:2051: RuntimeWarning: invalid value encountered in sqrt
  wmid=np.sqrt(w1*w2)
/home/sophie-stucki/starsim/starsim/spectra.py:2052: RuntimeWarning: invalid value encountered in log10
  dwave=(np.log10(w2)-np.log10(w1))/(npix-1.)
/home/sophie-stucki/starsim/starsim/spectra.py:2080: RuntimeWarning: invalid value encountered in sqrt
  wmid=np.sqrt(w1*w2)


Exception: The kernel can't be normalized, because its sum is close to zero. The sum of the given kernel is < 0.01. For a zero-sum kernel, set normalize_kernel=False or pass a custom normalization function to normalize_kernel.

In [9]:
ccf = ss_mps.results['ccf_quiet']

In [93]:
from scipy import interpolate
from astropy.convolution import convolve_fft 
from starsim import spectra

def vlambda(inlamb,vstep):
    #Resampler for convolution.
    # Short explanation: Second Clarke's Law
    # Long explanation: you need to resample stuff in a convinient spacing before convolving.
    xw2=max(inlamb)-0.1
    xw1=min(inlamb)+0.1
    iw1=np.where(inlamb > xw1)
    iw2=np.where(inlamb > xw2)
    iw1=iw1[0]
    iw2=iw2[0]
    w1=inlamb[iw1[0]+1]
    w2=inlamb[iw2[0]-1]


    npix=np.int64((iw2[0]-1)-(iw1[0]+1)*vstep)
    wmid=np.sqrt(w1*w2)

    dwave=(np.log10(w2)-np.log10(w1))/(npix-1.)
    vwavel=np.log10(wmid)-(dwave*np.arange(np.int64(npix/2.)))
    vwaveu=np.log10(wmid)+(dwave*np.arange(np.int64(npix/2.)))
    vwavel=10.**vwavel[1:len(vwavel)-1]
    vwavel=np.sort(vwavel)
    vwaveu=10.**vwaveu[1:len(vwaveu)-1]
    vwave=np.concatenate((vwavel,[wmid],vwaveu))
    return vwave


def add_resol(rv, ccf, instrument):
#"""
  # This function is mostly based on the SteParSyn broadener (Tabernero et al. 2022) 
  # SteParsyn is under the two-clause BSD licence, I added a disclaimer to take this into account
  # Input is wavelength in A, and flux in any unit. If input is in RV you should convert from RV to wavelength by assuming a central lambda (i.e. 6705.1 A).
        vstep=1
        vlight=2.99792458e5
        vwave=vlambda(rv,vstep)
        print(vwave)
        xw2=max(rv)-0.1
        xw1=min(rv)+0.1
        iw1=np.where(rv > xw1)
        iw2=np.where(rv > xw2)
        iw1=iw1[0]
        iw2=iw2[0]
        w1=rv[iw1[0]+1]
        w2=rv[iw2[0]-1]
        tck=interpolate.splrep(rv,ccf,k=3, s=0)
        vflux=interpolate.splev(vwave,tck,der=0)
        wmid=np.sqrt(w1*w2)

        x1=vwave
        y1=vflux

        if instrument == 'EXPRESS':
          Resolution = 137000.
          kop ='g'
        elif instrument == 'HARPS':
          Resolution = 115000.
          kop = 'g'
        elif instrument == 'HARPS-N':
          Resolution = 118000.
          kop = 'g'
        elif instrument == 'NEID':
          Resolution = 120000. 
          kop = 'g'
      
        if kop == 'g': 
            vibr=(vlight/Resolution)/(2.*np.sqrt(2.*np.log(2.)))
            sigma=vibr*wmid/vlight
            nx1=len(x1)
            dx1=(x1[nx1-1]-x1[0])/float(nx1-1)
            xk = (np.arange(nx1)-nx1/2)*dx1
            a1=0.
            a2=sigma
            zk=(xk-a1)/a2
            a0=1./np.sqrt(2.*np.pi)/sigma
            yk=a0*np.exp(-(zk**2.)/2.)

        nfact = np.sum(nx1)
        print(zk)
        outflux = convolve_fft(y1, yk/nfact, boundary='fill',fill_value=1.)  
  
        #flux in resampled back to he original sampling
        tck2 = interpolate.splrep(vwave,outflux,k=3, s=0)
        convolved_flux = interpolate.splev(rv, tck2, der=0)

        return convolved_flux


In [94]:
rv = (np.arange(-18,18.25,0.25) + 18)


add_resol(rv, ccf, 'HARPS-N')

[ 0.53144716  0.54790484  0.56487217  0.58236494  0.60039942  0.61899239
  0.63816113  0.65792349  0.67829784  0.69930314  0.72095892  0.74328534
  0.76630315  0.79003376  0.81449926  0.8397224   0.86572664  0.89253617
  0.92017593  0.94867162  0.97804977  1.00833768  1.03956354  1.07175639
  1.10494618  1.13916378  1.17444102  1.21081072  1.24830669  1.28696383
  1.32681809  1.36790655  1.41026742  1.4539401   1.49896523  1.54538467
  1.59324162  1.64258059  1.69344747  1.74588958  1.7999557   1.85569612
  1.91316269  1.97240887  2.03348976  2.09646218  2.16138471  2.22831775
  2.29732354  2.36846628  2.44181214  2.51742935  2.59538825  2.67576136
  2.75862343  2.84405155  2.93212518  3.02292624  3.1165392   3.21305113
  3.31255181  3.4151338   3.52089251  3.62992632  3.74233665  3.85822806
  3.97770837  4.1008887   4.22788363  4.3588113   4.4937935   4.63295578
  4.77642759  4.92434238  5.07683775  5.23405555  5.39614201  5.56324791
  5.73552869  5.91314461  6.09626087  6.28504782  6

Exception: The kernel can't be normalized, because its sum is close to zero. The sum of the given kernel is < 0.01. For a zero-sum kernel, set normalize_kernel=False or pass a custom normalization function to normalize_kernel.